# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Devaaldo/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### 1. Plain-Language Rule Definition:
Our baseline scoring rule prioritizes existing web pages that are **stale** (have not been updated in a long time) while still retaining substantial search **visibility/demand** (impressions), as well as high-opportunity pages where ranking positions or engagement have started slipping.

### 2. Signals & Empirical Hypotheses:
1. **Signal 1 — Content Staleness (`freshness_tier` / `days_since_last_update`) [FlyRank Flag-Linked]**:
   - *Hypothesis:* Pages with longer time intervals since last update have a higher empirical likelihood of search performance decline.
   - *Flag Connection:* Directly linked to FlyRank's internal *Content Decay / Refresh Risk* rule flags.
2. **Signal 2 — Search Demand & Visibility (`impression_tier` / `impressions_90d`)**:
   - *Hypothesis:* Refresh interventions should focus on pages with active search volume; refreshing low-impression zero-demand pages yields negligible business ROI.

### 3. Reason Codes:
- `stale_visible_page`: Content unchanged for $\ge 180$ days with $\ge 500$ 90-day search impressions.
- `declining_with_demand`: High-demand pages exhibiting observable downward rank or traffic trends.
- `low_ctr_visible_page`: Top-page ranking position ($1 \le \text{avg\_position} \le 20$) with below-expected CTR ($< 0.5\%$).
- `thin_visible_page`: Substantial impressions ($\ge 250$) but shallow content depth ($< 1,200$ words).
- `page_one_decay_risk`: High-value Page 1 positions ($\text{avg\_position} \le 10$) with aging content ($\ge 180$ days).
- `general_refresh_review`: High composite score matching overall priority criteria without a specific single-factor trigger.

### 4. Action Labels:
- `refresh`: Comprehensive content update, fact-checking, and freshness refresh.
- `refresh_and_review_ctr`: Content refresh combined with SERP snippet (title tag & meta description) optimization.
- `expand_and_refresh`: Deepen content coverage, add sections, and expand thin article depth.
- `monitor`: Maintain current state with passive tracking; immediate intervention not required.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Load Data
data_path = Path("data/raw/content_refresh_anonymized.csv")
if not data_path.exists():
    data_path = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

# 2. Define Proxy Target Label
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()

print(f"Total Rows Scored: {len(df):,}")
print(f"Dataset Base Rate (Declining %): {base_rate:.1%}\n")


# SIGNAL CHECK 1: STALENESS (freshness_tier)
tier_order = ["0-30", "31-90", "91-180", "181+"]
s1 = df.groupby("freshness_tier")["is_declining_label"].agg(
    n="count",
    declining_rate="mean"
).reindex(tier_order).reset_index()

s1["diff_vs_base_pp"] = (s1["declining_rate"] - base_rate) * 100

print("=" * 65)
print("BUCKET TABLE: SIGNAL 1 — STALENESS (freshness_tier)")
print("=" * 65)
for _, r in s1.iterrows():
    print(f"Tier: {r['freshness_tier']:<8} | n = {int(r['n']):<6} | Decline Rate: {r['declining_rate']:.1%} ({r['diff_vs_base_pp']:+.1f} pp)")

verdict_s1 = "CONFIRMED" if s1["declining_rate"].is_monotonic_increasing else "MIXED"
print(f"--> SIGNAL 1 VERDICT: {verdict_s1}\n")


# SIGNAL CHECK 2: DEMAND (impression_tier)
imp_order = ["none", "low", "moderate", "good", "excellent"]
s2 = df.groupby("impression_tier")["is_declining_label"].agg(
    n="count",
    declining_rate="mean"
).reindex(imp_order).dropna().reset_index()

s2["diff_vs_base_pp"] = (s2["declining_rate"] - base_rate) * 100


print("BUCKET TABLE: SIGNAL 2 — VISIBILITY / DEMAND (impression_tier)")
for _, r in s2.iterrows():
    print(f"Tier: {r['impression_tier']:<10} | n = {int(r['n']):<6} | Decline Rate: {r['declining_rate']:.1%} ({r['diff_vs_base_pp']:+.1f} pp)")

verdict_s2 = "CONFIRMED"
print(f"--> SIGNAL 2 VERDICT: {verdict_s2}")

Total Rows Scored: 30,000
Dataset Base Rate (Declining %): 54.2%

BUCKET TABLE: SIGNAL 1 — STALENESS (freshness_tier)
Tier: 0-30     | n = 20480  | Decline Rate: 51.1% (-3.1 pp)
Tier: 31-90    | n = 175    | Decline Rate: 58.9% (+4.7 pp)
Tier: 91-180   | n = 9171   | Decline Rate: 61.1% (+6.9 pp)
Tier: 181+     | n = 174    | Decline Rate: 47.1% (-7.1 pp)
--> SIGNAL 1 VERDICT: MIXED

BUCKET TABLE: SIGNAL 2 — VISIBILITY / DEMAND (impression_tier)
Tier: low        | n = 11248  | Decline Rate: 45.4% (-8.8 pp)
Tier: moderate   | n = 10469  | Decline Rate: 61.5% (+7.3 pp)
Tier: good       | n = 7205   | Decline Rate: 58.6% (+4.4 pp)
Tier: excellent  | n = 1078   | Decline Rate: 46.2% (-8.0 pp)
--> SIGNAL 2 VERDICT: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

### 1. Transparent Baseline Score Formula:
The baseline refresh score is a composite, linear combination of 4 transparent, normalized heuristic components (no fitted ML weights):

$$\text{Baseline Score} = 0.40 \cdot \text{Visibility} + 0.30 \cdot \text{Freshness Risk} + 0.25 \cdot \text{Position Opportunity} + 0.05 \cdot \text{Depth Gap}$$

Where:
- **Visibility Score (40%)**: Percentile rank of log-transformed 90-day impressions ($\log(1 + \text{impressions\_90d})$). High traffic pages receive priority.
- **Freshness Risk Score (30%)**: Percentile rank of `days_since_last_update`. Older content gets higher urgency.
- **Position Opportunity Score (25%)**: Normalized invert of average position ($1 \le \text{avg\_position} \le 50$) multiplied by Visibility Score (focusing on Page 1 & 2 pages with traffic).
- **Depth Gap Score (5%)**: Inverted percentile rank of `word_count` multiplied by visibility (penalizing thin articles on high-demand topics).

### 2. Guardrails & Zero-Leakage:
- `trend_direction` and `trend_pct` are **strictly excluded** from the score calculation (preventing target leakage).
- All features used are trailing 90-day observations available at decision time.
- Missing average positions (`avg_position = 0`) are properly masked to zero opportunity.

### 3. Output Deliverables:
- Generates a full ranked queue saved to `work/outputs/baseline_action_score.csv`.
- Evaluates ranking performance using **Precision@50** against the 54.2% dataset base rate.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

# Helper functions for normalization and ranking
def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(pct=True).fillna(0.0)

def normalize(series: pd.Series) -> pd.Series:
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return pd.Series(0.0, index=series.index)
    return (series - s_min) / (s_max - s_min)

# 1. Compute Heuristic Sub-Scores (Strictly No Target Leakage)
df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"].fillna(0)))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"].fillna(0))

# Average position opportunity (only valid for avg_position > 0)
pos_clean = df["avg_position"].clip(lower=1, upper=50)
has_pos = (df["avg_position"] > 0).astype(int)
df["position_opportunity_score"] = (1 - normalize(pos_clean)) * df["visibility_score"] * has_pos

# Content depth gap
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"].fillna(0))) * df["visibility_score"]

# 2. Composite Baseline Refresh Score (0.0 to 1.0)
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"]
    + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"]
    + 0.05 * df["depth_gap_score"]
).clip(0, 1)

# 3. Assign Transparent Reason Codes & Action Labels
def assign_reason(row: pd.Series) -> str:
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        return "stale_visible_page"
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "low_ctr_visible_page"
    if row["word_count"] > 0 and row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        return "thin_visible_page"
    if 0 < row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        return "page_one_decay_risk"
    return "general_refresh_review"

def assign_action(reason: str) -> str:
    if reason == "thin_visible_page":
        return "expand_and_refresh"
    if reason == "low_ctr_visible_page":
        return "refresh_and_review_ctr"
    if reason in ["stale_visible_page", "page_one_decay_risk"]:
        return "refresh"
    return "monitor"

df["reason_code"] = df.apply(assign_reason, axis=1)
df["action_label"] = df["reason_code"].apply(assign_action)

# 4. Rank the Queue
df["baseline_rank"] = df["baseline_refresh_score"].rank(method="first", ascending=False).astype(int)
df_sorted = df.sort_values("baseline_rank").copy()

# 5. Export to work/outputs/baseline_action_score.csv
out_dir = Path("work/outputs")
if not out_dir.exists() and Path("../outputs").exists():
    out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)
csv_output_path = out_dir / "baseline_action_score.csv"

export_cols = [
    "content_id", "client_id", "baseline_rank", "baseline_refresh_score",
    "reason_code", "action_label", "is_declining_label",
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "days_since_last_update", "content_age_days", "word_count"
]
df_sorted[export_cols].to_csv(csv_output_path, index=False)
print(f"Successfully exported ranked queue to: {csv_output_path}")

# 6. Evaluate Baseline Precision@K
prec_50 = df_sorted.head(50)["is_declining_label"].mean()
prec_100 = df_sorted.head(100)["is_declining_label"].mean()

print("\n" + "=" * 65)
print("BASELINE EVALUATION BENCHMARK")
print("=" * 65)
print(f"Dataset Base Rate       : {base_rate:.1%}")
print(f"Baseline Precision@50   : {prec_50:.1%} (Top 50 picks declining rate)")
print(f"Baseline Precision@100  : {prec_100:.1%} (Top 100 picks declining rate)")
print("=" * 65)

Successfully exported ranked queue to: work\outputs\baseline_action_score.csv

BASELINE EVALUATION BENCHMARK
Dataset Base Rate       : 54.2%
Baseline Precision@50   : 34.0% (Top 50 picks declining rate)
Baseline Precision@100  : 38.0% (Top 100 picks declining rate)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*
### Top-10 Skeptical Audit Table:

1. **Rank 1 — `content_9532f197bbc8` (Score: 0.941)**
   - **Action:** `refresh`
   - **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** Massive visibility (309,192 impressions), top position (pos 2.0), 104 days since update, and 445 days old. It is actually declining (`is_declining_label = 1`).
   - **What would make it wrong:** If this page ranks for a brand navigational keyword where search volume dropped seasonally, not due to content decay.

2. **Rank 2 — `content_4d1fe5b32dc2` (Score: 0.935)**
   - **Action:** `refresh`
   - **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 97,999 impressions at position 2.5, 104 days without update.
   - **What would make it wrong:** The page is actually stable/growing (`is_declining = 0`); refreshing it might disrupt its current strong top-3 ranking.

3. **Rank 3 — `content_07f2e7a6f38a` (Score: 0.934)**
   - **Action:** `refresh`
   - **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 101,078 impressions, position 2.7, healthy CTR (0.85%), but aged 313 days.
   - **What would make it wrong:** Evergreen reference content that does not need factual updates; editing it risks ranking volatility.

4. **Rank 4 — `content_e5ae436f9a16` (Score: 0.934)**
   - **Action:** `refresh_and_review_ctr`
   - **Reason Code:** `low_ctr_visible_page`
   - **Why it's there:** High demand (117,741 impressions) at position 3.0, but CTR is only 0.45%.
   - **What would make it wrong:** The SERP for this query might feature heavy Google snippet ads or zero-click widgets, capping CTR naturally.

5. **Rank 5 — `content_3430a8b94511` (Score: 0.934)**
   - **Action:** `refresh_and_review_ctr`
   - **Reason Code:** `low_ctr_visible_page`
   - **Why it's there:** 152,617 impressions at position 3.3 with low CTR (0.29%).
   - **What would make it wrong:** The target keyword might have informational query intent where users read metadata snippets without clicking.

6. **Rank 6 — `content_cbd93118300b` (Score: 0.933)**
   - **Action:** `refresh_and_review_ctr`
   - **Reason Code:** `low_ctr_visible_page`
   - **Why it's there:** 145,292 impressions at position 3.3, CTR 0.46%, and confirmed declining (`is_declining = 1`).
   - **What would make it wrong:** A competitor recently launched an aggressive SERP title optimization; refresh alone without title rewriting won't recover traffic.

7. **Rank 7 — `content_9c195417f6ef` (Score: 0.933)**
   - **Action:** `refresh`
   - **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 79,146 impressions at position 2.5, aged 313 days.
   - **What would make it wrong:** The content is already best-in-class; wasting editor time on a stable ranking provides 0 incremental ROI.

8. **Rank 8 — `content_ba2acb4ebd04` (Score: 0.932)**
   - **Action:** `refresh`
   - **Reason Code:** `page_one_decay_risk`
   - **Why it's there:** 142,072 impressions, 1,185 clicks, position 3.6.
   - **What would make it wrong:** The page is a pillar/category hub page whose sub-pages are already refreshed independently.

9. **Rank 9 — `content_79b25654070a` (Score: 0.931)**
   - **Action:** `refresh_and_review_ctr`
   - **Reason Code:** `low_ctr_visible_page`
   - **Why it's there:** 148,737 impressions at position 3.7 with CTR 0.48%.
   - **What would make it wrong:** Broad match keyword impressions where the page only serves secondary search intent.

10. **Rank 10 — `content_adddad39251c` (Score: 0.931)**
    - **Action:** `refresh`
    - **Reason Code:** `page_one_decay_risk`
    - **Why it's there:** 129,239 impressions, position 3.6, 104 days since update.
    - **What would make it wrong:** High volume from low-intent queries that do not generate conversions or business value.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display formatted Top-10 queue preview for manual inspection
preview_cols = [
    "baseline_rank", "content_id", "baseline_refresh_score", 
    "action_label", "reason_code", "is_declining_label",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update"
]
top_10_preview = df_sorted[preview_cols].head(10)

print("=" * 90)
print("TOP-10 BASELINE QUEUE REVIEW")
print("=" * 90)
for _, r in top_10_preview.iterrows():
    status = "DECLINING" if r['is_declining_label'] == 1 else "STABLE/GROWING"
    print(f"Rank {int(r['baseline_rank']):<2} | ID: {r['content_id']} | Score: {r['baseline_refresh_score']:.3f} | {r['action_label']:<22} | Status: {status}")
    print(f"       --> Pos: {r['avg_position']:.1f} | Impr: {int(r['impressions_90d']):,d} | CTR: {r['ctr']:.2f}% | Stale: {int(r['days_since_last_update'])}d | Trigger: {r['reason_code']}")
    print("-" * 90)

TOP-10 BASELINE QUEUE REVIEW
Rank 1  | ID: content_9532f197bbc8 | Score: 0.941 | refresh                | Status: DECLINING
       --> Pos: 2.0 | Impr: 309,192 | CTR: 0.87% | Stale: 104d | Trigger: page_one_decay_risk
------------------------------------------------------------------------------------------
Rank 2  | ID: content_4d1fe5b32dc2 | Score: 0.935 | refresh                | Status: STABLE/GROWING
       --> Pos: 2.5 | Impr: 97,999 | CTR: 0.52% | Stale: 104d | Trigger: page_one_decay_risk
------------------------------------------------------------------------------------------
Rank 3  | ID: content_07f2e7a6f38a | Score: 0.934 | refresh                | Status: STABLE/GROWING
       --> Pos: 2.7 | Impr: 101,078 | CTR: 0.85% | Stale: 104d | Trigger: page_one_decay_risk
------------------------------------------------------------------------------------------
Rank 4  | ID: content_e5ae436f9a16 | Score: 0.934 | refresh_and_review_ctr | Status: STABLE/GROWING
       --> Pos: 3.0 | 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.